In [36]:
import pandas as pd

df = pd.read_csv("auto-mpg.csv").drop(columns=["car name", "origin"]).rename(columns={"model year": "model_year"})
df = df[df["horsepower"] != "?"]
df["horsepower"] = df["horsepower"].astype(int)

df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year
0,18.0,8,307.0,130,3504,12.0,70
1,15.0,8,350.0,165,3693,11.5,70
2,18.0,8,318.0,150,3436,11.0,70
3,16.0,8,304.0,150,3433,12.0,70
4,17.0,8,302.0,140,3449,10.5,70


In [37]:
df.describe().drop("count").T

,mean,std,min,25%,50%,75%,max
mpg,23.445918,7.805007,9.0,17.000,22.75,29.000,46.6
cylinders,5.471939,1.705783,3.0,4.000,4.00,8.000,8.0
displacement,194.411990,104.644004,68.0,105.000,151.00,275.750,455.0
horsepower,104.469388,38.491160,46.0,75.000,93.50,126.000,230.0
weight,2977.584184,849.402560,1613.0,2225.250,2803.50,3614.750,5140.0
acceleration,15.541327,2.758864,8.0,13.775,15.50,17.025,24.8
model_year,75.979592,3.683737,70.0,73.000,76.00,79.000,82.0


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 392 entries, 0 to 397
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           392 non-null    float64
 1   cylinders     392 non-null    int64  
 2   displacement  392 non-null    float64
 3   horsepower    392 non-null    int64  
 4   weight        392 non-null    int64  
 5   acceleration  392 non-null    float64
 6   model_year    392 non-null    int64  
dtypes: float64(3), int64(4)
memory usage: 24.5 KB


In [39]:
X, y = df.drop(columns=["mpg"]), df["mpg"]
X.shape, y.shape

((392, 6), (392,))

In [40]:
from sklearn.model_selection import train_test_split

# chosen model = RandomForestRegressor
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((313, 6), (79, 6), (313,), (79,))

In [41]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

model = RandomForestRegressor()
model.fit(X_train,y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print(f"{mae=}")
print(f"{mse=}")
print(f"{rmse=}")

mae=1.7830253164556964
mse=6.381753556962022
rmse=2.5262132841393306


### acceleration feature included got better results, but removing origin also got better results

In [42]:
y_pred[:10]

array([25.825, 22.116, 34.562, 30.463, 27.117, 27.204, 12.79 , 27.758,
       19.464, 32.013])

In [43]:
y_test[:10].values

array([26. , 21.6, 36.1, 26. , 27. , 28. , 13. , 26. , 19. , 29. ])

In [44]:
import joblib

model.fit(X,y)

joblib.dump(model, "model/mpg_regressor.joblib", compress=("xz", 3), protocol=5)

['model/mpg_regressor.joblib']

In [45]:
model = joblib.load("model/mpg_regressor.joblib")

sample_test_data = pd.DataFrame(X_test.iloc[0]).T
sample_test_data

,cylinders,displacement,horsepower,weight,acceleration,model_year
79,4.0,96.0,69.0,2189.0,18.0,72.0


In [46]:
y_test.iloc[0]

np.float64(26.0)

In [47]:
model.predict(sample_test_data)

array([26.09])

In [48]:
feature_importance = pd.DataFrame([X.columns, model.feature_importances_])
feature_importance = feature_importance.T
feature_importance.columns = ["Feature", "Importance"]
feature_importance.sort_values(by= "Importance", ascending=False)

,Feature,Importance
1,displacement,0.31187
3,weight,0.219975
0,cylinders,0.205162
5,model_year,0.122906
2,horsepower,0.109469
4,acceleration,0.030618
